# 03 · 질환 라벨 — ICD-10

`metadata.csv`의 `icd10_truncated`가 환자의 진단코드 목록을 담고 있다. 세그먼트마다 반복되므로
환자당 한 번만 읽어 **환자 × 코드 이진 행렬**로 만든다.

1단계 분석 대상은 **환자 100명 이상인 3자리 코드**다. 대조군은 그 코드가 없는 나머지 전원.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import numpy as np
import pandas as pd

from ppg_fm import paths, features as F

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)
print("프로젝트", paths.ROOT)
print("데이터  ", paths.data_root())
from ppg_fm.report import Report
rep = Report("01_dataset/03_labels_icd")
print("산출물 →", rep.dir)

In [ ]:
from ppg_fm.data import labels

m_path = paths.interim("patient_icd_matrix_v2.csv")
if not m_path.exists():
    labels.build()
n = labels.code_counts()
print(f"코드 총수 {len(n):,} · 환자 ≥100명 {int((n>=100).sum())} · ≥50명 {int((n>=50).sum())}")

## 1. 사례 수 분포

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 3.2), dpi=140)
ax.hist(n[n > 0], bins=np.logspace(0, np.log10(n.max()), 50), color="#5E646F")
ax.set_xscale("log"); ax.axvline(100, color="#9E3B4A", lw=1.2)
ax.set_xlabel("patients per code (log)"); ax.set_ylabel("number of codes")
ax.set_title("ICD-10 3-digit codes by patient count (line = analysis cutoff, n=100)", fontsize=10)
rep.figure(fig, "icd_case_count_hist.png", "코드별 환자 수 분포")
plt.show()

## 2. 사례가 많은 코드

In [ ]:
top = n.head(25).rename("환자 수").to_frame()
top["비율(%)"] = (100 * top["환자 수"] / 6113).round(1)
rep.table(n.rename("n_patient").rename_axis("icd10").reset_index(), "icd_code_counts.csv", "코드별 환자 수 전수")
top

## 3. 순환계 코드 — 1단계의 주 관심

파형 특징이 혈역학을 반영한다면 이 코드들에서 먼저 나타나야 한다.

In [ ]:
circ = n[[c for c in n.index if c.startswith("I")]]
circ[circ >= 100].to_frame("환자 수")

## 4. 분석 대상 확정

In [ ]:
codes = sorted(n[n >= 100].index)
print(f"분석 대상 코드 {len(codes)}개")
print(f"검정 수 = {len(codes)} × {len(F.PATIENT)} = {len(codes)*len(F.PATIENT):,}")
rep.table(pd.DataFrame({"icd10": codes, "n_patient": n[codes].values}), "analysis_codes.csv", "환자 100명 이상 코드")

## 산출물

In [ ]:
rep.done("ICD-10 라벨 지형과 분석 대상 코드")
rep.summary()